#

### READ THE BRONZE DATA, CREATE DATA FRAME

In [14]:
customers_raw=spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/customers.parquet")
orders_raw=spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/orders.parquet")
payments_raw=spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/payments.parquet")
support_raw=spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/support_tickets.parquet")
web_raw=spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/web_activities.parquet")

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 16, Finished, Available, Finished, False)

### Read Bronze Data

In [15]:
display(customers_raw.limit(5))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9e8a93c0-3cea-4541-b0e4-9bd8318635b0)

In [16]:
display(orders_raw.limit(5))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d0e328d-09ad-43bd-b2e0-46e1fd270115)

In [17]:
display(payments_raw.limit(5))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0a4e3f9-d8a0-4f1e-9ce3-67c9c11abe98)

In [18]:
display(support_raw.limit(5))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5b07746-4a22-4d71-a1dd-7d1b1d9d9d71)

In [19]:
display(web_raw.limit(5))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 578d9f55-334e-47e8-9e68-0de98503a268)

# Save Bronze Data into delta tables

In [20]:
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 22, Finished, Available, Finished, False)

# Cleaning of data

#### Cleaned customer data

In [21]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

customers=spark.table('customers')

customers_clean = (
    customers
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("EMAIL", lower(trim(col("EMAIL"))))
    .withColumn(
        "gender",
        when(lower(col("gender")).isin("f", "female"), "Female")
        .when(lower(col("gender")).isin("m", "male"), "Male")
        .otherwise("Other")
    )
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(trim(col("location"))))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "EMAIL"])
)

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 23, Finished, Available, Finished, False)

In [22]:
display(customers_clean.limit(10))
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customer")

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 74303693-3651-4ea7-b2b6-d6380df62699)

# clean order data


In [23]:
order = spark.table("orders")
orders_clean = (
order.withColumn("order_date",when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"),to_date(col("order_date"),"yyyy/MM/dd"))
                            .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"),to_date(col("order_date"),"dd-MM-yyyy"))
                            .when(col("order_date").rlike("^\d{8}$"),to_date(col("order_date"),"yyyyMMdd"))
                            .otherwise(to_date(col("order_date"),"yyyy-MM-dd")))
       .withColumn("amount",col("amount").cast(DoubleType()))  
       .withColumn("amount",when(col("amount")<0,None).otherwise(col("amount")))
       .withColumn("status",initcap(trim(col("status"))))
       .dropna(subset=["customer_id","order_date"])
       .dropDuplicates(["order_id"])                  
)
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_order")
display(orders_clean.limit(10))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65eb82c5-22f4-4a2b-bcb1-db6142df55f8)

### Cleaning Payment data

In [24]:
payments = spark.table("payments")
payment_clean=(
   payments.withColumn("payment_date",when(col("payment_date").rlike("^\d{4}/\d{2}/\d{2}$"),to_date(col("payment_date"),"yyyy/MM/dd"))
                            .when(col("payment_date").rlike("^\d{2}-\d{2}-\d{4}$"),to_date(col("payment_date"),"dd-MM-yyyy"))
                            .when(col("payment_date").rlike("^\d{8}$"),to_date(col("payment_date"),"yyyyMMdd"))
                            .otherwise(to_date(col("payment_date"),"yyyy-MM-dd")))
            .withColumn("payment_method",initcap(trim(col("payment_method"))))
            .withColumn("payment_method",when(col("payment_method")=="Creditcard","Credit Card").otherwise(col("payment_method")))
            .withColumn("payment_status",initcap(trim(col("payment_status"))))
            .withColumn("amount",col("amount").cast(DoubleType()))    
            .withColumn("amount",when(col("amount")<0,None).otherwise(col("amount")))
            .dropna(subset=["customer_id","payment_date","amount"])
            .dropDuplicates(["payment_id"])           
)

payment_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")
display(payment_clean.limit(5))

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 673570bb-25b7-44b0-8690-469b1272654b)

In [25]:
payment_clean.groupBy("payment_method").count().show()

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 27, Finished, Available, Finished, False)

+--------------+-----+
|payment_method|count|
+--------------+-----+
|        Paypal|    2|
|   Credit Card|    4|
|    Netbanking|    1|
|           Upi|    2|
|          Cash|    1|
|        Wallet|    1|
+--------------+-----+



### Clean Support Data

In [26]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29219b78-4402-49b3-b68b-569fd4bef668)

### Clean Web Data

In [27]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")


StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e38ac30d-1566-4c3e-ac0c-56a2b3bb45cf)

### Gold Tables - Aggragrate Tables

In [28]:
cust = spark.table("silver_customer").alias("c")
orders = spark.table("silver_order").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 2918400b-965c-42c9-bba3-26697069adb9, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1276a19f-d116-4393-90dc-cd1937378ebc)